# Synthetic data generation -- Kaggle runner

Generates new Task 1 + Task 2 training data via controlled bug injection (see
[METHOD.md](https://github.com/ptran1203/msc/blob/main/llm/METHOD.md) section 4 for the
design: the LLM only writes buggy code from a precise instruction, labels come from an
objective oracle -- never from the LLM self-grading its own output).

Attach the private dataset `ptran1203/llm-autograding-course-data` under Notebook -> Add Input before running.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
%cd /kaggle/working
!rm -rf msc
!git clone --depth 1 https://github.com/ptran1203/msc.git
%cd /kaggle/working/msc/llm
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -y -qq g++ > /dev/null
!g++ --version


In [ ]:
import shutil, pathlib
dataset_root = pathlib.Path("/kaggle/input/datasets/ptran1203/llm-autograding-course-data")
repo_llm = pathlib.Path("/kaggle/working/msc/llm")
shutil.copytree(dataset_root / "data", repo_llm / "data")
print("data staged OK")


## Smoke test -- oracle self-check, then 1 bug type x 1 variant x 3 problems

Per the no-heavy-compute-without-checking-first rule: verify the oracle (compile/run/compare,
no model needed) and the full LLM pipeline both work before the real run.

In [ ]:
%cd /kaggle/working/msc/llm/src
!python synth_oracle.py


In [ ]:
!python synth_generate.py --smoke-test


## Full run -- 3 problems x 10 bug types x 3 variants (up to 90 candidates before QC discards)

Edit `--variants` to generate more/fewer per bug type.

In [ ]:
!python -u synth_generate.py --variants 3


## Output

Written to `/kaggle/working/msc/llm/synth/generated/` (schema-compatible with `sample_dataset`,
loadable via `data.load_dataset()`) -- saved as this notebook's output automatically on commit.

**Before using this data elsewhere**: it embeds real exam statements (course content), so per
the course rules it should go back into the private Kaggle dataset
(`ptran1203/llm-autograding-course-data`), never into the public code repo or anywhere public -- re-upload it there,
don't just leave it only as a kernel output.